In [ ]:
import logging
import os

import litellm
from agents import InputGuardrailTripwireTriggered, set_trace_processors
from IPython.display import Markdown, display
from opik.integrations.openai.agents import OpikTracingProcessor
from pydantic import ValidationError

from agent import GuardrailOutput, run_osint
from session import ensure_db
import time

In [ ]:
litellm.suppress_debug_info = True
logging.getLogger('mcp.os.posix.utilities').setLevel(logging.ERROR)

In [ ]:
os.environ['OPIK_PROJECT_NAME'] = 'osint-agent'
set_trace_processors(processors=[OpikTracingProcessor()])

await ensure_db()

SESSION_ID = f'demo_{time.time_ns()}'

In [ ]:
try:
    response = await run_osint('С кем встречается Елена Летучая?', session_id=SESSION_ID)
    display(Markdown(response))
except InputGuardrailTripwireTriggered as e:
    output = e.guardrail_result.output
    assert isinstance(output.output_info, GuardrailOutput)
    display(Markdown(output.output_info.rejection_reason))
except ValidationError as e:
    display(Markdown(str(e)))

In [ ]:
try:
    response = await run_osint('А где работает?', session_id=SESSION_ID)
    display(Markdown(response))
except InputGuardrailTripwireTriggered as e:
    output = e.guardrail_result.output
    assert isinstance(output.output_info, GuardrailOutput)
    display(Markdown(output.output_info.rejection_reason))
except ValidationError as e:
    display(Markdown(str(e)))

### Check guardrail

In [ ]:
try:
    response = await run_osint('Помоги решить уравнение: x^2 + x = 729.', enable_guardrail=True)
    display(Markdown(response))
except InputGuardrailTripwireTriggered as e:
    output = e.guardrail_result.output
    assert isinstance(output.output_info, GuardrailOutput)
    display(Markdown(output.output_info.rejection_reason))